# MLB Data Import to Fabric Lakehouse

This notebook imports MLB CSV data into the lakehouse.
Make sure CSV files are uploaded to the lakehouse Files section first.


In [ ]:
# Import libraries
import pandas as pd
from pyspark.sql import SparkSession

# Initialize Spark session
spark = SparkSession.builder.appName('MLB_Data_Import').getOrCreate()

print('✅ Spark session initialized')

In [ ]:
# 1. Import Teams Data
print('📊 Importing teams data...')

teams_df = spark.read.option('header', True).csv('/lakehouse/default/Files/teams.csv')
teams_df.write.mode('overwrite').saveAsTable('teams')

print(f'✅ Imported {teams_df.count()} teams')
teams_df.show(5)

In [ ]:
# 2. Import Players Data
print('👥 Importing players data...')

players_df = spark.read.option('header', True).csv('/lakehouse/default/Files/players.csv')
players_df.write.mode('overwrite').saveAsTable('players')

print(f'✅ Imported {players_df.count()} players')
players_df.show(5)

In [ ]:
# 3. Import Games Data
print('🎮 Importing games data...')

games_df = spark.read.option('header', True).csv('/lakehouse/default/Files/games.csv')
games_df.write.mode('overwrite').saveAsTable('games')

print(f'✅ Imported {games_df.count()} games')
games_df.show(5)

In [ ]:
# 4. Import Boxscore Data (with Enhanced Statistics!)
print('📊 Importing boxscore data with stolen bases and caught stealing...')

boxscore_df = spark.read.option('header', True).csv('/lakehouse/default/Files/boxscore.csv')
boxscore_df.write.mode('overwrite').saveAsTable('boxscore')

print(f'✅ Imported {boxscore_df.count()} boxscore entries')
print('🔍 Enhanced statistics included:')
print('   - stolen_bases column')
print('   - caught_stealing column')
boxscore_df.show(5)

In [ ]:
# 5. Verify Enhanced Statistics
print('🔍 Verifying stolen bases and caught stealing data...')

# Check for non-zero stolen bases
stolen_bases_count = boxscore_df.filter(boxscore_df.stolen_bases > 0).count()
caught_stealing_count = boxscore_df.filter(boxscore_df.caught_stealing > 0).count()

print(f'📈 Players with stolen bases: {stolen_bases_count}')
print(f'📈 Players caught stealing: {caught_stealing_count}')

if stolen_bases_count > 0:
    print('🏃 Players with stolen bases:')
    boxscore_df.filter(boxscore_df.stolen_bases > 0).select('player_id', 'stolen_bases').show()

if caught_stealing_count > 0:
    print('🚫 Players caught stealing:')
    boxscore_df.filter(boxscore_df.caught_stealing > 0).select('player_id', 'caught_stealing').show()

In [ ]:
# 6. Data Summary
print('📊 Final Data Summary')
print('=' * 30)

# Count records in each table
tables = ['teams', 'players', 'games', 'boxscore']

for table in tables:
    try:
        df = spark.table(table)
        count = df.count()
        print(f'📈 {table}: {count:,} records')
    except Exception as e:
        print(f'❌ {table}: Error - {e}')

print('\n✅ MLB data successfully imported to lakehouse!')
print('🎯 Enhanced statistics (stolen bases, caught stealing) are now available!')